# 01 — Data Exploration

Exploratory data analysis of the synthetic histopathology tile dataset.

We examine:
1. Class distribution and imbalance
2. Sample tile visualization
3. Per-class pixel statistics
4. Why accuracy is misleading for this dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os, glob

In [ ]:
# Load manifest and show class distribution
manifest = pd.read_csv("../data/manifest.csv")
print(f"Total tiles: {len(manifest)}")
print(f'Splits: {manifest["split"].value_counts().to_dict()}')
print()

class_counts = manifest["label"].value_counts()
class_pcts = manifest["label"].value_counts(normalize=True) * 100
print("Class distribution:")
for cls in ["tumor", "immune", "stroma", "necrosis"]:
    print(f"  {cls:>10s}: {class_counts[cls]:4d} ({class_pcts[cls]:.1f}%)")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#8B45A6", "#4169E1", "#E8A0BF", "#D3D3D3"]
bars = ax.bar(["tumor", "immune", "stroma", "necrosis"],
              [class_counts.get(c, 0) for c in ["tumor", "immune", "stroma", "necrosis"]],
              color=colors)
for bar, cls in zip(bars, ["tumor", "immune", "stroma", "necrosis"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f"{class_counts[cls]}", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Count")
ax.set_title("Class Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize sample tiles: 4 tiles per class in a 4x4 grid
classes = ["tumor", "immune", "stroma", "necrosis"]
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for row, cls in enumerate(classes):
    cls_files = manifest[manifest["label"] == cls]["filepath"].values
    selected = np.random.choice(cls_files, size=min(4, len(cls_files)), replace=False)
    for col, fpath in enumerate(selected):
        img = Image.open(os.path.join("../data", fpath))
        arr = np.array(img)
        axes[row, col].imshow(arr)
        axes[row, col].set_title(f"{cls}\n{arr.shape} {arr.dtype}", fontsize=10)
        axes[row, col].axis("off")

plt.suptitle("Sample Tiles (4 per class)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Pixel statistics per class
stats = []
for cls in classes:
    cls_files = manifest[manifest["label"] == cls]["filepath"].values
    sample_files = np.random.choice(cls_files, size=min(50, len(cls_files)), replace=False)
    pixels = []
    for fpath in sample_files:
        img = np.array(Image.open(os.path.join("../data", fpath))).astype(np.float32)
        pixels.append(img)
    pixels = np.stack(pixels)
    for ch, ch_name in enumerate(["R", "G", "B"]):
        stats.append({
            "class": cls,
            "channel": ch_name,
            "mean": pixels[:, :, :, ch].mean(),
            "std": pixels[:, :, :, ch].std(),
        })

stats_df = pd.DataFrame(stats)
pivot = stats_df.pivot_table(index="class", columns="channel", values=["mean", "std"])
print("Pixel Statistics per Class (0-255 range):")
print(pivot.round(1))

In [ ]:
# Class imbalance analysis
majority_count = class_counts.max()
minority_count = class_counts.min()
imbalance_ratio = majority_count / minority_count

majority_class = class_counts.idxmax()
majority_pct = class_pcts[majority_class]

print(f"Imbalance ratio (majority/minority): {imbalance_ratio:.1f}:1")
print(f"Majority class: {majority_class} ({majority_pct:.1f}%)")
print()
print(f'A model that ALWAYS predicts "{majority_class}" would achieve:')
print(f"  Accuracy = {majority_pct:.1f}%  <-- misleading!")
print(f"  Macro-F1 = {1/len(classes)*100:.1f}%  <-- exposes the problem")
print()
print("Why accuracy is misleading:")
print("  Accuracy counts ALL correct predictions equally. In an imbalanced dataset,")
print("  a model can achieve high accuracy by simply predicting the majority class.")
print(f"  With {majority_pct:.0f}% stroma, always predicting stroma gives ~{majority_pct:.0f}% accuracy.")
print()
print("  Macro-F1 computes F1 for EACH class independently, then averages.")
print("  A majority-class-only model gets 0 F1 on tumor/immune/necrosis,")
print("  resulting in a low macro-F1 that correctly reflects poor performance.")